# Build an Interactive Dashboard with Plotly Dash
This notebook builds a SpaceX Launch Records Dashboard with:
- **Task 1**: Launch Site Drop-down Input Component
- **Task 2**: Pie chart callback for success rates
- **Task 3**: Payload Range Slider
- **Task 4**: Scatter chart callback for payload vs. success

In [1]:
# Install required libraries (uncomment if needed)
# !pip install dash plotly pandas

In [2]:
import pandas as pd
import dash
from dash import html
from dash import dcc
from dash.dependencies import Input, Output
import plotly.express as px

In [3]:
# Read the SpaceX launch data into a pandas DataFrame
spacex_df = pd.read_csv("spacex_launch_dash.csv")

# Get payload range for the slider
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

print(f"Payload range: {min_payload} kg — {max_payload} kg")
print(f"Available launch sites: {spacex_df['Launch Site'].unique().tolist()}")
spacex_df.head()

Payload range: 0.0 kg — 9600.0 kg
Available launch sites: ['CCAFS LC-40', 'VAFB SLC-4E', 'KSC LC-39A', 'CCAFS SLC-40']


,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [4]:
# Create the Dash application
app = dash.Dash(__name__)

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# APP LAYOUT
# ─────────────────────────────────────────────────────────────────────────────
app.layout = html.Div(children=[

    # Dashboard Title
    html.H1(
        'SpaceX Launch Records Dashboard',
        style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}
    ),

    # ── TASK 1: Launch Site Drop-down ────────────────────────────────────────
    # Allows the user to filter by a specific launch site or view all sites.
    # Default value is 'ALL' so the dashboard loads with aggregate data.
    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites',    'value': 'ALL'},
            {'label': 'CCAFS LC-40',  'value': 'CCAFS LC-40'},
            {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'},
            {'label': 'KSC LC-39A',   'value': 'KSC LC-39A'},
            {'label': 'VAFB SLC-4E',  'value': 'VAFB SLC-4E'},
        ],
        value='ALL',                          # default: show all sites
        placeholder='Select a Launch Site here',
        searchable=True                       # allow typing to filter options
    ),

    html.Br(),

    # ── TASK 2: Pie Chart placeholder ────────────────────────────────────────
    # When 'ALL' is selected → proportion of successes per site.
    # When a specific site is selected → Success vs. Failure for that site.
    html.Div(dcc.Graph(id='success-pie-chart')),

    html.Br(),

    html.P("Payload range (Kg):"),

    # ── TASK 3: Payload Range Slider ─────────────────────────────────────────
    # Lets users filter launches by payload mass (0 – 10 000 kg).
    # Initial range spans the full dataset extent.
    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={
            0:     '0',
            2500:  '2500',
            5000:  '5000',
            7500:  '7500',
            10000: '10000',
        },
        value=[min_payload, max_payload]      # initial handle positions
    ),

    html.Br(),

    # ── TASK 4: Scatter Chart placeholder ────────────────────────────────────
    # Shows payload mass vs. launch outcome, coloured by booster version.
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),

])

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 2 CALLBACK — Pie Chart
# Input  : site-dropdown  (value)
# Output : success-pie-chart (figure)
# ─────────────────────────────────────────────────────────────────────────────
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown',      component_property='value')
)
def get_pie_chart(entered_site):
    """
    ALL sites  → pie slices = total successful launches per site.
    One site   → pie slices = Success (1) vs. Failure (0) for that site.
    """
    if entered_site == 'ALL':
        # Sum 'class' (1 = success) per launch site to get success counts
        fig = px.pie(
            spacex_df,
            values='class',
            names='Launch Site',
            title='Total Successful Launches by Site'
        )
    else:
        # Filter to the selected site
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]

        # Count outcomes and give them readable labels
        outcome_counts = (
            filtered_df['class']
            .value_counts()
            .reset_index()
        )
        # Rename columns for clarity
        outcome_counts.columns = ['class', 'count']
        # Map numeric class to human-readable labels
        outcome_counts['Outcome'] = outcome_counts['class'].map(
            {1: 'Success', 0: 'Failure'}
        )

        fig = px.pie(
            outcome_counts,
            values='count',
            names='Outcome',
            color='Outcome',
            color_discrete_map={'Success': '#2ecc71', 'Failure': '#e74c3c'},
            title=f'Launch Outcomes for {entered_site}'
        )

    return fig

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK 4 CALLBACK — Scatter Chart
# Inputs : site-dropdown (value), payload-slider (value)
# Output : success-payload-scatter-chart (figure)
# ─────────────────────────────────────────────────────────────────────────────
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [
        Input(component_id='site-dropdown',  component_property='value'),
        Input(component_id='payload-slider', component_property='value'),
    ]
)
def get_scatter_chart(entered_site, payload_range):
    """
    Plots Payload Mass (kg) on the x-axis vs. launch class (0/1) on y-axis.
    Points are coloured by Booster Version Category.
    Filtered first by the slider's payload range, then optionally by site.
    """
    low, high = payload_range

    # Apply payload range filter
    mask = spacex_df['Payload Mass (kg)'].between(low, high)
    df_filtered = spacex_df[mask]

    if entered_site == 'ALL':
        fig = px.scatter(
            df_filtered,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title='Payload vs. Launch Outcome for All Sites',
            labels={'class': 'Launch Outcome (1=Success, 0=Failure)'},
            hover_data=['Launch Site']
        )
    else:
        # Further filter to the selected launch site
        df_site = df_filtered[df_filtered['Launch Site'] == entered_site]

        fig = px.scatter(
            df_site,
            x='Payload Mass (kg)',
            y='class',
            color='Booster Version Category',
            title=f'Payload vs. Launch Outcome for {entered_site}',
            labels={'class': 'Launch Outcome (1=Success, 0=Failure)'}
        )

    # Make y-axis ticks human-readable
    fig.update_yaxes(
        tickvals=[0, 1],
        ticktext=['Failure (0)', 'Success (1)']
    )

    return fig

In [8]:
# Run the Dash application
# jupyter_mode='inline' renders the app inside the notebook
if __name__ == '__main__':
    app.run(jupyter_mode='inline', debug=True)

---
## Quiz Answers

**Question 3** — Which attribute provides available selections for a Plotly Dropdown input?  
> **`options`** — a list of `{'label': ..., 'value': ...}` dicts.

**Question 4** — How is a callback result associated with a layout element?  
> Using a **unique `component_id`** shared between the layout element and the `Output(...)` in the callback decorator.

**Question 5** — Can multiple input components be added to a single callback?  
> **Yes** — wrap them in a list: `[Input(...), Input(...)]`.